In [1]:
import numpy as np
from tensorflow.keras import layers, models


In [2]:
data_run1 = np.load("C:/Users/Sanjana/Downloads/Run355456_Dataset_jqkne.npy")
data_run2 = np.load("C:/Users/Sanjana/Downloads/Run357479_Dataset_iodic.npy")

print("Run1 shape:", data_run1.shape)
print("Run2 shape:", data_run2.shape)
print("Run1 dtype:", data_run1.dtype)
print("Run2 dtype:", data_run2.dtype)


Run1 shape: (10000, 64, 72)
Run2 shape: (10000, 64, 72)
Run1 dtype: float64
Run2 dtype: float64


In [3]:
X_train = data_run1 / np.max(data_run1)
X_test = data_run2 / np.max(data_run2) 


In [4]:
y_train = np.zeros(len(X_train))  # Run A = 0
y_test = np.ones(len(X_test))     # Run B = 1


In [5]:
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("Final train shape:", X_train.shape)
print("Final test shape:", X_test.shape)


Final train shape: (10000, 64, 72, 1)
Final test shape: (10000, 64, 72, 1)


In [6]:
# Per-image normalization
X_train = X_train / (X_train.max(axis=(1,2), keepdims=True) + 1e-8)
X_test  = X_test  / (X_test.max(axis=(1,2), keepdims=True) + 1e-8)


In [7]:

from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), padding='same', input_shape=(64,72,1)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])


C:\Users\Sanjana\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [9]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)


Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 58s 158ms/step - accuracy: 0.9778 - loss: 0.0900 - val_accuracy: 0.0000e+00 - val_loss: 177.2192
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 48s 151ms/step - accuracy: 1.0000 - loss: 0.0055 - val_accuracy: 0.0000e+00 - val_loss: 464.9175
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 48s 152ms/step - accuracy: 1.0000 - loss: 0.0020 - val_accuracy: 0.0000e+00 - val_loss: 549.3405
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 47s 151ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.0000e+00 - val_loss: 524.1708
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 48s 152ms/step - accuracy: 1.0000 - loss: 7.3290e-04 - val_accuracy: 0.0000e+00 - val_loss: 572.0873


In [10]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Cross-run accuracy (BatchNorm):", test_acc)


313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.0000e+00 - loss: 572.0873
Cross-Run Accuracy: 0.0
